# Matmul综合实践

通过本章的系统学习，我们已经掌握了使用Ascend C开发算子、进行Host侧调用和运行验证的基础方法。为了巩固矩阵乘高阶API和Kernel直调流程，现提供以下实践练习：实现Ascend C算子Matmul，算子命名为matmul_custom。

相关算法：

                                  C = A * B + Bias

支持M = 1024，K = 256，N = 640的int8类型输入、int32类型Bias和输出，算子原型为：

<div style="width: 100%;">
<table style="border-collapse: collapse; width: 550px; max-width: 550px; border: 1px solid #ddd; margin-left: 0 !important;">
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子核函数名</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" > matmul_custom</td>
  </tr>
  <tr>
    <td rowspan="4" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">a</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(1024, 256)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">int8</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">b</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(256, 640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">int8</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">bias</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">int32</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">c</td>
    <td style="padding: 8px;">(1024, 640)</td>
    <td style="padding: 8px;">int32</td>
    <td style="padding: 8px;">ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>

要求：

- 完成matmul_custom算子Host侧Tiling生成逻辑。
- 完成matmul_custom算子Kernel侧Matmul高阶API调用逻辑。
- 完成Host侧调用 `matmul_custom<<<BLOCK_DIM, nullptr, stream>>>(...)` 
- 支持输入类型为int8，Bias和输出类型为int32。

输入A矩阵全为1，B矩阵全为2，Bias全为10。预期输出C矩阵全为522。

---

## 环境准备


In [ ]:
!mkdir -p Sources/03.06

import os
import subprocess
from pathlib import Path

set_env = os.environ.get("ASCEND_TOOLKIT_HOME", "/usr/local/Ascend/cann") + "/set_env.sh"
if not Path(set_env).exists():
    set_env = "/usr/local/Ascend/cann/set_env.sh"

result = subprocess.run(
    ["bash", "-lc", f"source {set_env} && env"],
    capture_output=True,
    text=True,
    check=True,
)
for line in result.stdout.strip().split("\n"):
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value

print("Environment initialization process completed successfully.")

---

## Host侧Tiling生成

补齐 `GenMatmulTiling` 函数，设置A、B、C、Bias的数据类型与矩阵形状，并生成Kernel侧使用的 `TCubeTiling`。


In [ ]:
%%writefile Sources/03.06/matmul_custom_tiling.h

#ifndef MATMUL_CUSTOM_TILING_H
#define MATMUL_CUSTOM_TILING_H

#include <cstdint>
#include <iostream>

#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"

inline bool GenMatmulTiling(platform_ascendc::PlatformAscendC* ascendcPlatform, TCubeTiling& tiling,
                            int32_t m, int32_t n, int32_t k, int32_t numBlocks)
{
    if (ascendcPlatform == nullptr) {
        std::cerr << "ascendcPlatform is nullptr." << std::endl;
        return false;
    }

    matmul_tiling::MultiCoreMatmulTiling cubeTiling(*ascendcPlatform);
    // TODO: TCubeTiling根据输入参数刷新切分策略

    return true;
}

#endif


---

## Kernel与Host直调实现

补齐 `matmul_custom.asc`：

- Kernel侧创建 `Matmul` 高阶API对象，绑定GlobalTensor并执行 `IterateAll`。
- Host侧完成ACL初始化、内存申请、数据拷贝、`<<<>>>` Kernel直调、结果回拷和资源释放。


In [ ]:
%%writefile Sources/03.06/matmul_custom.asc

#include <algorithm>
#include <cstdint>
#include <iostream>
#include <iterator>
#include <vector>

#include "acl/acl.h"
#include "kernel_operator.h"
#define ASCENDC_CUBE_ONLY
#include "lib/matmul_intf.h"
#include "matmul_custom_tiling.h"

using namespace matmul;

constexpr int32_t M = 1024;
constexpr int32_t K = 256;
constexpr int32_t N = 640;
constexpr int32_t BLOCK_DIM = 1;

__global__ __cube__ void matmul_custom(__gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c,
    __gm__ uint8_t* workspace, AscendC::tiling::TCubeTiling tiling)
{
    // TODO: Kernel实现
}

std::vector<int32_t> kernel_matmul(std::vector<int8_t>& a, std::vector<int8_t>& b,
                                   std::vector<int32_t>& bias)
{
    // TODO: 调用matmul_custom
    return {};
}

uint32_t VerifyResult(std::vector<int32_t>& output, std::vector<int32_t>& golden)
{
    auto printTensor = [](std::vector<int32_t>& tensor, const char* name) {
        constexpr size_t maxPrintSize = 10;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
                  std::ostream_iterator<int32_t>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };

    printTensor(output, "Output");
    printTensor(golden, "Golden");
    if (output.size() == golden.size() && std::equal(golden.begin(), golden.end(), output.begin())) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    }
    std::cout << "[Failed] Case accuracy is verification failed!" << std::endl;
    return 1;
}

int32_t main(int32_t argc, char* argv[])
{
    int8_t valueA = 1;
    int8_t valueB = 2;
    int32_t valueBias = 10;
    int32_t valueOutput = 522;

    std::vector<int8_t> a(M * K, valueA);
    std::vector<int8_t> b(K * N, valueB);
    std::vector<int32_t> bias(N, valueBias);

    std::vector<int32_t> output = kernel_matmul(a, b, bias);
    std::vector<int32_t> golden(M * N, valueOutput);
    return VerifyResult(output, golden);
}

---

## CMake编译配置

该部分无需修改，直接执行即可。


In [ ]:
%%writefile Sources/03.06/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
set(CMAKE_ASC_ARCHITECTURES "dav-2201" CACHE STRING "NPU architecture: dav-2201, dav-3510")
find_package(ASC REQUIRED)
project(matmul_practice LANGUAGES ASC CXX)

add_executable(demo
    matmul_custom.asc
)

target_link_libraries(demo PRIVATE
    tiling_api
    register
    platform
    m
    dl
)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)


---

## 编译和运行

修改完成后，执行如下命令编译并运行用例。


In [ ]:
!cd Sources/03.06 && \
mkdir -p build && \
cd build && \
cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 .. && \
make -j && \
./demo


预期执行效果如下：

```cpp
Output: 522 522 522 522 522 522 522 522 522 522 ...
Golden: 522 522 522 522 522 522 522 522 522 522 ...
[Success] Case accuracy is verification passed.
```

---

执行以下代码获取答案。


In [ ]:
!cat ./answer/03.06/matmul_custom.asc

In [ ]:
!cat ./answer/03.06/matmul_custom_tiling.h